<a href="https://colab.research.google.com/github/singhpremshankar921/bioinformatics-tools/blob/main/RNAseq_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nproc
!free -h
!df -h

2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       652Mi       8.9Gi       2.0Mi       3.1Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       114G   22G   93G  19% /kaggle/input
tmpfs           6.4G   32K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [ ]:
!apt-get update -qq
!apt-get install -y -qq sra-toolkit fastqc multiqc samtools

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
(Reading database ... 119547 files and directories currently installed.)
Preparing to unpack .../000-libpython3.10-dev_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../001-libpython3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../002-python3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../003-libpython3.10-stdlib_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../004-python3

In [ ]:
!apt-get update -qq
!apt-get install -y sra-toolkit

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
sra-toolkit is already the newest version (2.11.3+dfsg-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 74 not upgraded.


In [ ]:
!prefetch --version
!fasterq-dump --version


"prefetch" version 2.11.3


"fasterq-dump" version 2.11.3



In [ ]:
!mkdir -p RNAseq_Project/{data/raw_fastq,data/trimmed_fastq,reference,alignment,counts,qc,results,scripts}

In [ ]:
!find RNAseq_Project -maxdepth 2 -type d

RNAseq_Project
RNAseq_Project/scripts
RNAseq_Project/alignment
RNAseq_Project/data
RNAseq_Project/data/raw_fastq
RNAseq_Project/data/trimmed_fastq
RNAseq_Project/reference
RNAseq_Project/results
RNAseq_Project/counts
RNAseq_Project/qc


In [ ]:
!prefetch SRR29928471 --output-directory RNAseq_Project/data/sra


2026-09-07T12:22:54 prefetch.2.11.3: Current preference is set to retrieve SRA Normalized Format files with full base quality scores.
2026-09-07T12:22:55 prefetch.2.11.3: 1) Downloading 'SRR29928471'...
2026-09-07T12:22:55 prefetch.2.11.3: SRA Normalized Format file is being retrieved, if this is different from your preference, it may be due to current file availability.
2026-09-07T12:22:55 prefetch.2.11.3:  Downloading via HTTPS...
2026-09-07T12:23:43 prefetch.2.11.3:  HTTPS download succeed
2026-09-07T12:23:56 prefetch.2.11.3:  'SRR29928471' is valid
2026-09-07T12:23:56 prefetch.2.11.3: 1) 'SRR29928471' was downloaded successfully
2026-09-07T12:23:56 prefetch.2.11.3: 'SRR29928471' has 0 unresolved dependencies


In [ ]:
ls -lh RNAseq_Project/data/sra/SRR29928471/

total 2.1G
-rw-r--r-- 1 root root 2.1G Sep  7 12:23 SRR29928471.sra


In [ ]:
!fasterq-dump SRR29928471 \
  --split-files \
  --threads 2 \
  -O RNAseq_Project/data/raw_fastq

2026-09-07T12:35:49 fasterq-dump.2.11.3 err: fasterq-dump.c fastdump_csra() checking ouput-file 'RNAseq_Project/data/raw_fastq/SRR29928471.fastq' -> RC(libs/kfs/unix/sysdir.c:2039:KSysDirOpenFileWrite_v1 rcExe,rcFile,rcPacking,rcName,rcExists)
fasterq-dump quit with error code 3


In [ ]:
!ls -lh RNAseq_Project/data/raw_fastq/

total 19G
-rw-r--r-- 1 root root 9.4G Sep  7 12:35 SRR29928471_1.fastq
-rw-r--r-- 1 root root 9.4G Sep  7 12:35 SRR29928471_2.fastq


In [ ]:

!mkdir -p RNAseq_Project/qc/raw

!fastqc \
  RNAseq_Project/data/raw_fastq/SRR29928471_1.fastq \
  RNAseq_Project/data/raw_fastq/SRR29928471_2.fastq \
  -o RNAseq_Project/qc/raw \
  -t 2

[0.033s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.033s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Started analysis of SRR29928471_1.fastq
Started analysis of SRR29928471_2.fastq
Approx 5% complete for SRR29928471_1.fastq
Approx 5% complete for SRR29928471_2.fastq
Approx 10% complete for SRR29928471_2.fastq
Approx 10% complete for SRR29928471_1.fastq
Approx 15% complete for SRR29928471_2.fastq
Approx 15% complete for SRR29928471_1.fastq
Approx 20% complete for SRR29928471_2.fastq
Approx 20% complete for SRR29928471_1.fastq
Approx 25% complete for SRR29928471_2.fastq
Approx 25% complete for SRR29928471_1.fastq
Approx 30% complete for SRR29928471_2.fastq
Approx 30% complete for SRR29928471_1.fastq
Approx 35% complete for SRR29928471_2.fastq
Approx 35% complete f

In [ ]:
!ls -lh RNAseq_Project/qc/raw/

total 2.8M
-rw-r--r-- 1 root root 599K Sep  7 12:49 SRR29928471_1_fastqc.html
-rw-r--r-- 1 root root 776K Sep  7 12:49 SRR29928471_1_fastqc.zip
-rw-r--r-- 1 root root 607K Sep  7 12:49 SRR29928471_2_fastqc.html
-rw-r--r-- 1 root root 785K Sep  7 12:49 SRR29928471_2_fastqc.zip


In [ ]:
!multiqc RNAseq_Project/qc/raw -o RNAseq_Project/qc/multiqc_raw

/usr/lib/python3/dist-packages/multiqc/multiqc.py:387: SyntaxWarning: invalid escape sequence '\.'
  if version.StrictVersion(re.sub("[^0-9\.]", "", remote_version)) > version.StrictVersion(
/usr/lib/python3/dist-packages/multiqc/multiqc.py:388: SyntaxWarning: invalid escape sequence '\.'
  re.sub("[^0-9\.]", "", config.short_version)
/usr/lib/python3/dist-packages/multiqc/multiqc.py:516: SyntaxWarning: invalid escape sequence '\w'
  filename = re.sub("[^\w\.-]", "", re.sub("[-\s]+", "-", title)).strip()
/usr/lib/python3/dist-packages/multiqc/multiqc.py:516: SyntaxWarning: invalid escape sequence '\s'
  filename = re.sub("[^\w\.-]", "", re.sub("[-\s]+", "-", title)).strip()
/usr/lib/python3/dist-packages/multiqc/utils/mqc_colour.py:28: SyntaxWarning: invalid escape sequence '\.'
  minval = re.sub("[^0-9\.-e]", "", str(minval))
/usr/lib/python3/dist-packages/multiqc/utils/mqc_colour.py:29: SyntaxWarning: invalid escape sequence '\.'
  maxval = re.sub("[^0-9\.-e]", "", str(maxval))
/usr/

In [ ]:
!ls -lh RNAseq_Project/qc/multiqc_raw/

total 1.2M
drwxr-xr-x 2 root root 4.0K Sep  7 12:50 multiqc_data
-rw-r--r-- 1 root root 1.2M Sep  7 12:50 multiqc_report.html


In [ ]:
!apt-get update -qq
!apt-get install -y -qq fastp

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fastp.
(Reading database ... 167945 files and directories currently installed.)
Preparing to unpack .../fastp_0.20.1+dfsg-1_amd64.deb ...
Unpacking fastp (0.20.1+dfsg-1) ...
Setting up fastp (0.20.1+dfsg-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!fastp --version

fastp 0.20.1


In [ ]:
!mkdir -p RNAseq_Project/data/trimmed_fastq

In [ ]:
!fastp \
-i RNAseq_Project/data/raw_fastq/SRR29928471_1.fastq \
-I RNAseq_Project/data/raw_fastq/SRR29928471_2.fastq \
-o RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq \
-O RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq \
--detect_adapter_for_pe \
--thread 2 \
-h RNAseq_Project/qc/fastp_report.html \
-j RNAseq_Project/qc/fastp_report.json

Detecting adapter sequence for read1...
>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 22622351
total bases: 3393352650
Q20 bases: 3331612055(98.1805%)
Q30 bases: 3218793856(94.8559%)

Read2 before filtering:
total reads: 22622351
total bases: 3393352650
Q20 bases: 3296755508(97.1533%)
Q30 bases: 3119835418(91.9396%)

Read1 after filtering:
total reads: 22437612
total bases: 3354390272
Q20 bases: 3300994095(98.4082%)
Q30 bases: 3191912804(95.1563%)

Read2 aftering filtering:
total reads: 22437612
total bases: 3354408610
Q20 bases: 3269348047(97.4642%)
Q30 bases: 3097003873(92.3264%)

Filtering result:
reads passed filter: 44875224
reads failed due to low quality: 365856
reads failed due to too many N: 3138
reads failed due to too short: 484
reads with adapter trimmed: 873754
bases trimmed due to adapters: 22654092

Duplicati

In [ ]:
!mkdir -p RNAseq_Project/qc/trimmed_fastqc

In [ ]:
!fastqc \
RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq \
RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq \
-o RNAseq_Project/qc/trimmed_fastqc \
-t 2

[0.018s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.018s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Started analysis of SRR29928471_1_trimmed.fastq
Started analysis of SRR29928471_2_trimmed.fastq
Approx 5% complete for SRR29928471_1_trimmed.fastq
Approx 5% complete for SRR29928471_2_trimmed.fastq
Approx 10% complete for SRR29928471_1_trimmed.fastq
Approx 10% complete for SRR29928471_2_trimmed.fastq
Approx 15% complete for SRR29928471_1_trimmed.fastq
Approx 15% complete for SRR29928471_2_trimmed.fastq
Approx 20% complete for SRR29928471_1_trimmed.fastq
Approx 20% complete for SRR29928471_2_trimmed.fastq
Approx 25% complete for SRR29928471_1_trimmed.fastq
Approx 25% complete for SRR29928471_2_trimmed.fastq
Approx 30% complete for SRR29928471_1_trimmed.fastq
Appro

In [ ]:
!ls RNAseq_Project/qc/trimmed_fastqc

SRR29928471_1_trimmed_fastqc.html  SRR29928471_2_trimmed_fastqc.html
SRR29928471_1_trimmed_fastqc.zip   SRR29928471_2_trimmed_fastqc.zip


In [ ]:
!multiqc RNAseq_Project/qc/trimmed_fastqc \
-o RNAseq_Project/qc/multiqc_trimmed


  /// ]8;id=935761;https://multiqc.info\MultiQC]8;;\ 🔍 | v1.12

|           multiqc | MultiQC Version v1.35 now available!
|           multiqc | Search path : /content/RNAseq_Project/qc/trimmed_fastqc
|         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 4/4  
/usr/local/lib/python3.13/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
|            fastqc | Found 2 reports
|           multiqc | Compressing plot data
|           multiqc | Report      : RNAseq_Project/qc/multiqc_trimmed/multiqc_report.html
|           multiqc | Data        : RNAseq_Project/qc/multiqc_trimmed/multiqc_data
|           multiqc | MultiQC complete


In [ ]:
!ls RNAseq_Project/qc/multiqc_trimmed

multiqc_data  multiqc_report.html


In [ ]:
!ls RNAseq_Project/qc/multiqc_trimmed

multiqc_data  multiqc_report.html


In [ ]:
from google.colab import files

files.download("RNAseq_Project/qc/multiqc_trimmed/multiqc_report.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("RNAseq_Project/qc/multiqc_trimmed/multiqc_report.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cat RNAseq_Project/qc/multiqc_trimmed/multiqc_data/multiqc_fastqc.txt | head -50

Sample	Filename	File type	Encoding	Total Sequences	Sequences flagged as poor quality	Sequence length	%GC	total_deduplicated_percentage	avg_sequence_length	basic_statistics	per_base_sequence_quality	per_tile_sequence_quality	per_sequence_quality_scores	per_base_sequence_content	per_sequence_gc_content	per_base_n_content	sequence_length_distribution	sequence_duplication_levels	overrepresented_sequences	adapter_content
SRR29928471_1	SRR29928471_1_trimmed.fastq	Conventional base calls	Sanger / Illumina 1.9	22437612.0	0.0	15-150	51.0	46.15105776854071	149.49919536891895	pass	pass	pass	pass	fail	pass	pass	warn	fail	pass	pass
SRR29928471_2	SRR29928471_2_trimmed.fastq	Conventional base calls	Sanger / Illumina 1.9	22437612.0	0.0	15-150	51.0	47.29444313481966	149.50002745390196	pass	pass	pass	pass	fail	pass	pass	warn	fail	pass	pass


In [ ]:
!mkdir -p RNAseq_Project/reference
!mkdir -p RNAseq_Project/star_index
!mkdir -p RNAseq_Project/alignment

In [ ]:
!ls RNAseq_Project

alignment  data  reference  runinfo.csv  star_index
counts	   qc	 results    scripts


In [ ]:
!cd RNAseq_Project/reference && \
wget -c https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/GRCh38.primary_assembly.genome.fa.gz && \
wget -c https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.primary_assembly.annotation.gtf.gz

--2026-09-07 14:08:51--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/GRCh38.primary_assembly.genome.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 845635028 (806M) [application/x-gzip]
Saving to: ‘GRCh38.primary_assembly.genome.fa.gz’

GRCh38.primary_asse 100%[===================>] 806.46M  3.33MB/s    in 5m 24s  

2026-09-07 14:14:16 (2.49 MB/s) - ‘GRCh38.primary_assembly.genome.fa.gz’ saved [845635028/845635028]

--2026-09-07 14:14:16--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.primary_assembly.annotation.gtf.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 124650284 (119M) [application/x-gzip]
Saving to: ‘gencode.v50.primary_asse

In [ ]:
!ls -lh /content/

total 267M
drwxr-xr-x  2 root root 4.0K Sep  7 14:24 bin
drwxr-xr-x 10 root root 4.0K Sep  7 14:07 RNAseq_Project
drwxr-xr-x  1 root root 4.0K Aug 24 13:21 sample_data
drwxrwxr-x  6 root root 4.0K Sep  7 11:55 sratoolkit
drwxrwxr-x  5 root root 4.0K Mar 25 16:47 sratoolkit.3.4.1-ubuntu64
-rw-r--r--  1 root root  86M Mar 25 16:47 sratoolkit.current-ubuntu64.tar.gz
-rw-r--r--  1 root root  86M Mar 25 16:47 sratoolkit.current-ubuntu64.tar.gz.1
-rw-r--r--  1 root root  86M Mar 25 16:47 sratoolkit.current-ubuntu64.tar.gz.2
drwxrwxr-x  7 root root 4.0K Jan 25  2024 STAR-2.7.11b
-rw-r--r--  1 root root  12M Sep  7 14:28 STAR.tar.gz
drwx------  2 root root 4.0K Sep  7 14:35 _STARtmp
drwxr-xr-x  2 root root 4.0K Sep  7 13:40 trimmed_qc


In [ ]:
!find /content/RNAseq_Project -maxdepth 3 -type f 2>/dev/null | head -50

/content/RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.progress.out
/content/RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.out
/content/RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam
/content/RNAseq_Project/data/raw_fastq/SRR29928471_2.fastq
/content/RNAseq_Project/data/raw_fastq/SRR29928471_1.fastq
/content/RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq
/content/RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq
/content/RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf
/content/RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa.gz
/content/RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf.gz
/content/RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa
/content/RNAseq_Project/qc/fastp_report.html
/content/RNAseq_Project/qc/multiqc_raw/multiqc_report.html
/content/RNAseq_Project/qc/multiqc_trimmed/multiqc_report.html
/content/RNAseq_Project/qc/trimmed_fastqc/SRR29928471_1_trim

In [ ]:
!ls -lh /content/STAR-2.7.11b/source/STAR

-rwxr-xr-x 1 root root 3.1M Sep  7 14:33 /content/STAR-2.7.11b/source/STAR


In [ ]:
!ls -lh RNAseq_Project/star_index 2>/dev/null

total 314M
-rw-r--r-- 1 root root 1.2K Sep  7 14:38 chrLength.txt
-rw-r--r-- 1 root root 3.2K Sep  7 14:38 chrNameLength.txt
-rw-r--r-- 1 root root 2.0K Sep  7 14:38 chrName.txt
-rw-r--r-- 1 root root 2.1K Sep  7 14:38 chrStart.txt
-rw-r--r-- 1 root root 175M Sep  7 14:38 exonGeTrInfo.tab
-rw-r--r-- 1 root root  73M Sep  7 14:38 exonInfo.tab
-rw-r--r-- 1 root root 3.2M Sep  7 14:38 geneInfo.tab
-rw-r--r-- 1 root root  26K Sep  7 14:39 Log.out
-rw-r--r-- 1 root root  22M Sep  7 14:38 sjdbList.fromGTF.out.tab
-rw-r--r-- 1 root root  42M Sep  7 14:38 transcriptInfo.tab


In [ ]:
!ls -lh RNAseq_Project/data/trimmed_fastq/

total 19G
-rw-r--r-- 1 root root 9.3G Sep  7 13:15 SRR29928471_1_trimmed.fastq
-rw-r--r-- 1 root root 9.3G Sep  7 13:15 SRR29928471_2_trimmed.fastq


In [ ]:
!ls -lh RNAseq_Project/alignment/SRR29928471/

total 8.0K
-rw-r--r-- 1 root root    0 Sep  7 14:42 SRR29928471_Aligned.out.bam
-rw-r--r-- 1 root root 2.7K Sep  7 14:42 SRR29928471_Log.out
-rw-r--r-- 1 root root    0 Sep  7 14:42 SRR29928471_Log.progress.out
drwx------ 2 root root 4.0K Sep  7 14:42 SRR29928471__STARtmp


In [ ]:
!tail -30 RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.out

###### All USER parameters from Command Line:
genomeDir                     RNAseq_Project/star_index     ~RE-DEFINED
readFilesIn                   RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq   RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq        ~RE-DEFINED
runThreadN                    2     ~RE-DEFINED
outFileNamePrefix             RNAseq_Project/alignment/SRR29928471/SRR29928471_     ~RE-DEFINED
outSAMtype                    BAM   Unsorted        ~RE-DEFINED
quantMode                     GeneCounts        ~RE-DEFINED
##### Finished reading parameters from all sources

##### Final user re-defined parameters-----------------:
runThreadN                        2
genomeDir                         RNAseq_Project/star_index
readFilesIn                       RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq   RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq   
outFileNamePrefix                 RNAseq_Project/alignment/SRR29928471/

In [ ]:
!ls -lh RNAseq_Project/star_index/genomeParameters.txt

In [92]:
!find RNAseq_Project -type f \( -name "*.fa" -o -name "*.fasta" -o -name "*.gtf" -o -name "*.gtf.gz" \)

RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf
RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf.gz
RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa


In [93]:
!find RNAseq_Project/reference/GRCh38 -type f -maxdepth 2 -printf '%p\n'

find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
find: ‘RNAseq_Project/reference/GRCh38’: No such file or directory


In [94]:
!ls -lh RNAseq_Project/reference/GRCh38

ls: cannot access 'RNAseq_Project/reference/GRCh38': No such file or directory


In [95]:
!ls -lh RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa

-rw-r--r-- 1 root root 3.0G Jun  9 12:55 RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa


In [96]:
!/content/STAR-2.7.11b/source/STAR \
--runMode genomeGenerate \
--genomeDir RNAseq_Project/star_index \
--genomeFastaFiles RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa \
--sjdbGTFfile RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf \
--sjdbOverhang 149 \
--runThreadN 2

	/content/STAR-2.7.11b/source/STAR --runMode genomeGenerate --genomeDir RNAseq_Project/star_index --genomeFastaFiles RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa --sjdbGTFfile RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf --sjdbOverhang 149 --runThreadN 2
	STAR version: 2.7.11b   compiled: 2026-09-07T14:29:19+00:00 488cd18412d8:/content/STAR-2.7.11b/source
Sep 07 15:09:12 ..... started STAR run
Sep 07 15:09:12 ... starting to generate Genome files
Sep 07 15:10:53 ..... processing annotations GTF
Sep 07 15:12:36 ... starting to sort Suffix Array. This may take a long time...
Sep 07 15:12:58 ... sorting Suffix Array chunks and saving them to disk...
^C


In [97]:
!/content/STAR-2.7.11b/source/STAR \
--genomeDir RNAseq_Project/star_index \
--readFilesIn \
RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq \
RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq \
--runThreadN 2 \
--outFileNamePrefix RNAseq_Project/alignment/SRR29928471/SRR29928471_ \
--outSAMtype BAM Unsorted \
--quantMode GeneCounts

	/content/STAR-2.7.11b/source/STAR --genomeDir RNAseq_Project/star_index --readFilesIn RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq --runThreadN 2 --outFileNamePrefix RNAseq_Project/alignment/SRR29928471/SRR29928471_ --outSAMtype BAM Unsorted --quantMode GeneCounts
	STAR version: 2.7.11b   compiled: 2026-09-07T14:29:19+00:00 488cd18412d8:/content/STAR-2.7.11b/source
Sep 07 15:13:54 ..... started STAR run
Sep 07 15:13:54 ..... loading genome

EXITING because of FATAL ERROR: could not open genome file RNAseq_Project/star_index//genomeParameters.txt
SOLUTION: check that the path to genome files, specified in --genomeDir is correct and the files are present, and have user read permsissions

Sep 07 15:13:54 ...... FATAL ERROR, exiting


In [99]:
!ls -lh RNAseq_Project/alignment/SRR29928471/

total 8.0K
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Aligned.out.bam
-rw-r--r-- 1 root root 2.7K Sep  7 15:13 SRR29928471_Log.out
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Log.progress.out
drwx------ 2 root root 4.0K Sep  7 15:13 SRR29928471__STARtmp


In [100]:
!cat RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.out

STAR version=2.7.11b
STAR compilation time,server,dir=2026-09-07T14:29:19+00:00 488cd18412d8:/content/STAR-2.7.11b/source
STAR git: 
##### Command Line:
/content/STAR-2.7.11b/source/STAR --genomeDir RNAseq_Project/star_index --readFilesIn RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq --runThreadN 2 --outFileNamePrefix RNAseq_Project/alignment/SRR29928471/SRR29928471_ --outSAMtype BAM Unsorted --quantMode GeneCounts
##### Initial USER parameters from Command Line:
outFileNamePrefix                 RNAseq_Project/alignment/SRR29928471/SRR29928471_
###### All USER parameters from Command Line:
genomeDir                     RNAseq_Project/star_index     ~RE-DEFINED
readFilesIn                   RNAseq_Project/data/trimmed_fastq/SRR29928471_1_trimmed.fastq   RNAseq_Project/data/trimmed_fastq/SRR29928471_2_trimmed.fastq        ~RE-DEFINED
runThreadN                    2     ~RE-DEFINED
outFileNamePrefix            

In [101]:
!samtools sort -@ 2 \
RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam \
-o RNAseq_Project/alignment/SRR29928471/SRR29928471_sorted.bam

samtools sort: failed to read header from "RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam"


In [102]:
!ls -lh RNAseq_Project/alignment/SRR29928471/

total 8.0K
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Aligned.out.bam
-rw-r--r-- 1 root root 2.7K Sep  7 15:13 SRR29928471_Log.out
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Log.progress.out
drwx------ 2 root root 4.0K Sep  7 15:13 SRR29928471__STARtmp


In [105]:
!which samtools && samtools --version | head -1

/usr/bin/samtools
samtools 1.13


In [106]:
!ls -lh RNAseq_Project/alignment/SRR29928471/

total 8.0K
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Aligned.out.bam
-rw-r--r-- 1 root root 2.7K Sep  7 15:13 SRR29928471_Log.out
-rw-r--r-- 1 root root    0 Sep  7 15:13 SRR29928471_Log.progress.out
drwx------ 2 root root 4.0K Sep  7 15:13 SRR29928471__STARtmp


In [107]:
!samtools quickcheck -v RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam

RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam was not identified as sequence data.
RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam


In [108]:
!ls -lh RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam

-rw-r--r-- 1 root root 0 Sep  7 15:13 RNAseq_Project/alignment/SRR29928471/SRR29928471_Aligned.out.bam


In [110]:
!tail -30 RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.progress.out

In [111]:
!tail -30 RNAseq_Project/alignment/SRR29928471/SRR29928471_Log.progress.out

In [112]:
!/content/STAR-2.7.11b/source/STAR \
--runMode genomeGenerate \
--genomeDir RNAseq_Project/star_index \
--genomeFastaFiles RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa \
--sjdbGTFfile RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf \
--sjdbOverhang 149 \
--genomeSAindexNbases 12 \
--genomeSAsparseD 2 \
--runThreadN 2

	/content/STAR-2.7.11b/source/STAR --runMode genomeGenerate --genomeDir RNAseq_Project/star_index --genomeFastaFiles RNAseq_Project/reference/GRCh38.primary_assembly.genome.fa --sjdbGTFfile RNAseq_Project/reference/gencode.v50.primary_assembly.annotation.gtf --sjdbOverhang 149 --genomeSAindexNbases 12 --genomeSAsparseD 2 --runThreadN 2
	STAR version: 2.7.11b   compiled: 2026-09-07T14:29:19+00:00 488cd18412d8:/content/STAR-2.7.11b/source
Sep 07 15:23:34 ..... started STAR run
Sep 07 15:23:34 ... starting to generate Genome files
Sep 07 15:25:03 ..... processing annotations GTF
Sep 07 15:26:37 ... starting to sort Suffix Array. This may take a long time...
Sep 07 15:26:50 ... sorting Suffix Array chunks and saving them to disk...
^C
